In [3]:
!ls -R /kaggle/input

/kaggle/input:
datasets

/kaggle/input/datasets:
kaisuke4u

/kaggle/input/datasets/kaisuke4u:
qwen-dist

/kaggle/input/datasets/kaisuke4u/qwen-dist:
04_evaluate_visual.py  train_verified.jsonl  utils.py


In [4]:
!cp /kaggle/input/datasets/kaisuke4u/qwen-dist/* /kaggle/working/

In [5]:
!ls -lh /kaggle/working/train_verified.jsonl
!ls -lh /kaggle/working/utils.py
!ls -lh /kaggle/working/04_evaluate_visual.py

-rw-r--r-- 1 root root 3.1M Jul  7 16:41 /kaggle/working/train_verified.jsonl
-rw-r--r-- 1 root root 1003 Jul  7 16:41 /kaggle/working/utils.py
-rw-r--r-- 1 root root 11K Jul  7 16:41 /kaggle/working/04_evaluate_visual.py


In [6]:
!ls /kaggle/working

04_evaluate_visual.py  train_verified.jsonl  utils.py


In [7]:
!nvidia-smi

Tue Jul  7 16:42:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
!pip uninstall -y torchao
!pip install -q --upgrade git+https://github.com/huggingface/transformers.git
!pip install -q --upgrade accelerate peft bitsandbytes safetensors tensorboard datasets

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 118.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 26.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0

RESTART

In [9]:
import torch, transformers, peft, datasets
print("GPUs:", torch.cuda.device_count())
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Imports OK")

GPUs: 2
Transformers: 5.14.0.dev0
PEFT: 0.19.1
Imports OK


In [10]:
%load_ext tensorboard
%tensorboard --logdir runs

<IPython.core.display.Javascript object>

In [11]:
from pathlib import Path

code = r'''
#!/usr/bin/env python3
import argparse
import inspect
import json
import os
import random
from dataclasses import dataclass
from typing import List, Dict, Any

import torch
from datasets import Dataset
from tqdm import tqdm

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    set_seed,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)


SYSTEM_PROMPT = (
    "You are a meticulous math tutor. Solve the problem step by step, "
    "ending with '#### <answer>'."
)


def is_main_process() -> bool:
    return int(os.environ.get("RANK", "0")) == 0


def load_jsonl(path: str) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def encode_example(ex: Dict[str, Any], tokenizer, max_seq_len: int):
    question = ex["question"].strip()
    answer = ex["teacher_solution"].strip()

    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Problem: {question}\n\nSolve it step by step."},
    ]

    full_messages = prompt_messages + [
        {"role": "assistant", "content": answer},
    ]

    prompt_ids = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True,
    )

    full_ids = tokenizer.apply_chat_template(
        full_messages,
        tokenize=True,
        add_generation_prompt=False,
    )

    if len(full_ids) > max_seq_len:
        full_ids = full_ids[:max_seq_len]

    prompt_len = min(len(prompt_ids), len(full_ids))
    labels = [-100] * prompt_len + full_ids[prompt_len:]

    if all(x == -100 for x in labels):
        return None

    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
        "length": len(full_ids),
    }


@dataclass
class CausalLMCollator:
    pad_token_id: int
    pad_to_multiple_of: int = 8

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)

        if self.pad_to_multiple_of:
            remainder = max_len % self.pad_to_multiple_of
            if remainder:
                max_len += self.pad_to_multiple_of - remainder

        input_ids = []
        attention_mask = []
        labels = []

        for f in features:
            pad_len = max_len - len(f["input_ids"])

            input_ids.append(f["input_ids"] + [self.pad_token_id] * pad_len)
            attention_mask.append(f["attention_mask"] + [0] * pad_len)
            labels.append(f["labels"] + [-100] * pad_len)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def make_training_args(args):
    params = inspect.signature(TrainingArguments.__init__).parameters

    kwargs = {
        "output_dir": args.output_dir,
        "per_device_train_batch_size": args.batch_size,
        "gradient_accumulation_steps": args.grad_accum,
        "num_train_epochs": args.num_train_epochs,
        "learning_rate": args.lr,
        "lr_scheduler_type": "cosine",
        "warmup_ratio": args.warmup_ratio,
        "logging_steps": args.logging_steps,
        "logging_first_step": True,
        "logging_dir": args.logging_dir,
        "save_strategy": "steps",
        "save_steps": args.save_steps,
        "save_total_limit": args.save_total_limit,
        "load_best_model_at_end": False,
        "do_eval": False,
        "fp16": True,
        "bf16": False,
        "gradient_checkpointing": True,
        "optim": "paged_adamw_8bit",
        "max_grad_norm": 1.0,
        "remove_unused_columns": False,
        "group_by_length": True,
        "length_column_name": "length",
        "dataloader_num_workers": args.num_workers,
        "report_to": args.report_to,
        "run_name": args.run_name,
    }

    if "eval_strategy" in params:
        kwargs["eval_strategy"] = "no"
    elif "evaluation_strategy" in params:
        kwargs["evaluation_strategy"] = "no"

    if "gradient_checkpointing_kwargs" in params:
        kwargs["gradient_checkpointing_kwargs"] = {"use_reentrant": False}

    if int(os.environ.get("WORLD_SIZE", "1")) > 1 and "ddp_find_unused_parameters" in params:
        kwargs["ddp_find_unused_parameters"] = False

    clean_kwargs = {k: v for k, v in kwargs.items() if k in params}
    return TrainingArguments(**clean_kwargs)


def main():
    parser = argparse.ArgumentParser()

    parser.add_argument("--base_model", type=str, default="Qwen/Qwen3.5-0.8B")
    parser.add_argument("--data", type=str, default="train_verified.jsonl")
    parser.add_argument("--output_dir", type=str, default="qlora_6000_fast")

    parser.add_argument("--num_train_epochs", type=float, default=1.0)
    parser.add_argument("--batch_size", type=int, default=2)
    parser.add_argument("--grad_accum", type=int, default=4)
    parser.add_argument("--max_seq_len", type=int, default=768)

    parser.add_argument("--lr", type=float, default=2e-4)
    parser.add_argument("--warmup_ratio", type=float, default=0.03)

    parser.add_argument("--lora_r", type=int, default=16)
    parser.add_argument("--lora_alpha", type=int, default=32)
    parser.add_argument("--lora_dropout", type=float, default=0.05)

    parser.add_argument("--logging_steps", type=int, default=5)
    parser.add_argument("--save_steps", type=int, default=100)
    parser.add_argument("--save_total_limit", type=int, default=3)

    parser.add_argument("--logging_dir", type=str, default="runs")
    parser.add_argument("--report_to", type=str, default="tensorboard")
    parser.add_argument("--run_name", type=str, default="qwen35_08b_gsm8k_6000_fast")

    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--max_samples", type=int, default=None)
    parser.add_argument("--num_workers", type=int, default=2)
    parser.add_argument("--resume_from_checkpoint", type=str, default=None)

    args = parser.parse_args()

    set_seed(args.seed)
    random.seed(args.seed)

    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    world_size = int(os.environ.get("WORLD_SIZE", "1"))

    if torch.cuda.is_available():
        torch.cuda.set_device(local_rank)

    if is_main_process():
        print("=" * 70)
        print("Fast Visual QLoRA Training")
        print("=" * 70)
        print(f"Base model      : {args.base_model}")
        print(f"Data            : {args.data}")
        print(f"Output          : {args.output_dir}")
        print(f"World size      : {world_size}")
        print(f"Local rank      : {local_rank}")
        print(f"Epochs          : {args.num_train_epochs}")
        print(f"Batch/GPU       : {args.batch_size}")
        print(f"Grad accum      : {args.grad_accum}")
        print(f"Effective batch : {args.batch_size * args.grad_accum * world_size}")
        print(f"Max seq len     : {args.max_seq_len}")
        print(f"TensorBoard dir : {args.logging_dir}")
        print("=" * 70)

    tokenizer = AutoTokenizer.from_pretrained(
        args.base_model,
        trust_remote_code=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "right"

    test_template = tokenizer.apply_chat_template(
        [
            {"role": "user", "content": "What is 6*7?"},
            {"role": "assistant", "content": "42"},
        ],
        tokenize=False,
    )

    if "<|im_start|>assistant" not in test_template:
        raise RuntimeError("Assistant chat template marker not found. Check tokenizer template.")

    rows = load_jsonl(args.data)
    random.shuffle(rows)

    if args.max_samples is not None:
        rows = rows[: args.max_samples]

    if is_main_process():
        print(f"Loaded rows: {len(rows)}")
        print("Tokenizing...")

    encoded = []
    skipped = 0

    iterator = tqdm(rows, desc="tokenizing") if is_main_process() else rows

    for row in iterator:
        item = encode_example(row, tokenizer, args.max_seq_len)
        if item is None:
            skipped += 1
        else:
            encoded.append(item)

    if not encoded:
        raise RuntimeError("No valid training examples after tokenization.")

    train_ds = Dataset.from_list(encoded)

    if is_main_process():
        lengths = [x["length"] for x in encoded]
        print(f"Tokenized examples : {len(encoded)}")
        print(f"Skipped examples   : {skipped}")
        print(f"Min length         : {min(lengths)}")
        print(f"Avg length         : {sum(lengths) / len(lengths):.1f}")
        print(f"Max length         : {max(lengths)}")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    device_map = {"": local_rank} if torch.cuda.is_available() else None

    model = AutoModelForCausalLM.from_pretrained(
        args.base_model,
        quantization_config=bnb_config,
        device_map=device_map,
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )

    try:
        model = prepare_model_for_kbit_training(
            model,
            use_gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
        )
    except TypeError:
        model = prepare_model_for_kbit_training(model)

    model.config.use_cache = False

    lora_config = LoraConfig(
        r=args.lora_r,
        lora_alpha=args.lora_alpha,
        lora_dropout=args.lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
    )

    model = get_peft_model(model, lora_config)

    if is_main_process():
        model.print_trainable_parameters()

    collator = CausalLMCollator(
        pad_token_id=tokenizer.pad_token_id,
        pad_to_multiple_of=8,
    )

    training_args = make_training_args(args)

    trainer_kwargs = {
        "model": model,
        "args": training_args,
        "train_dataset": train_ds,
        "data_collator": collator,
    }

    trainer_params = inspect.signature(Trainer.__init__).parameters

    if "processing_class" in trainer_params:
        trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in trainer_params:
        trainer_kwargs["tokenizer"] = tokenizer

    trainer = Trainer(**trainer_kwargs)

    trainer.train(resume_from_checkpoint=args.resume_from_checkpoint)

    trainer.save_model(args.output_dir)

    if trainer.is_world_process_zero():
        tokenizer.save_pretrained(args.output_dir)
        print(f"\nFinal adapter saved to: {args.output_dir}")
        print(f"TensorBoard logs saved to: {args.logging_dir}")


if __name__ == "__main__":
    main()
'''

Path("/kaggle/working/02_train_fast_visual.py").write_text(code, encoding="utf-8")
print("Created /kaggle/working/02_train_fast_visual.py")

Created /kaggle/working/02_train_fast_visual.py


In [12]:
from pathlib import Path

path = Path("/kaggle/working/02_train_fast_visual.py")
text = path.read_text(encoding="utf-8")

start = text.index("def encode_example")
end = text.index("@dataclass")

new_encode = r'''
def encode_example(ex: Dict[str, Any], tokenizer, max_seq_len: int):
    question = str(ex.get("question", "")).strip()
    answer = str(ex.get("teacher_solution", "")).strip()

    if not question or not answer:
        return None

    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Problem: {question}\n\nSolve it step by step."},
    ]

    # Build prompt text first.
    # This contains system + user + assistant-start marker.
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # Full training text = prompt + teacher answer + EOS.
    eos = tokenizer.eos_token or ""
    full_text = prompt_text + answer + eos

    prompt_ids = tokenizer(
        prompt_text,
        add_special_tokens=False,
    )["input_ids"]

    full_ids = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=max_seq_len,
    )["input_ids"]

    prompt_len = len(prompt_ids)

    # If the prompt alone fills the whole context, skip this example.
    if prompt_len >= len(full_ids):
        return None

    labels = [-100] * prompt_len + full_ids[prompt_len:]

    # Safety check: at least one answer token must be trainable.
    if all(x == -100 for x in labels):
        return None

    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
        "length": len(full_ids),
    }


'''

text = text[:start] + new_encode + text[end:]
path.write_text(text, encoding="utf-8")

print("Patched encode_example successfully.")

Patched encode_example successfully.


In [13]:
!python -m py_compile /kaggle/working/02_train_fast_visual.py

**FULL TRAINING**

In [14]:
!rm -rf qlora_6000_fast
!CUDA_VISIBLE_DEVICES=0,1 accelerate launch \
  --multi_gpu \
  --num_processes=2 \
  --num_machines=1 \
  --mixed_precision=fp16 \
  --dynamo_backend=no \
  02_train_fast_visual.py \
  --data train_verified.jsonl \
  --output_dir qlora_6000_fast \
  --num_train_epochs 1 \
  --batch_size 2 \
  --grad_accum 4 \
  --max_seq_len 768 \
  --lr 1e-4 \
  --logging_steps 5 \
  --save_steps 100 \
  --save_total_limit 3 \
  --report_to tensorboard \
  --run_name qwen35_08b_gsm8k_6000_1epoch

Fast Visual QLoRA Training
Base model      : Qwen/Qwen3.5-0.8B
Data            : train_verified.jsonl
Output          : qlora_6000_fast
World size      : 2
Local rank      : 0
Epochs          : 1.0
Batch/GPU       : 2
Grad accum      : 4
Effective batch : 16
Max seq len     : 768
TensorBoard dir : runs
config.json: 2.91kB [00:00, 10.2MB/s]
tokenizer_config.json: 16.7kB [00:00, 35.6MB/s]
vocab.json: 6.72MB [00:00, 122MB/s]
merges.txt: 3.35MB [00:00, 124MB/s]
tokenizer.json: 100%|██████████████████████| 12.8M/12.8M [00:00<00:00, 29.3MB/s]
chat_template.jinja: 7.75kB [00:00, 17.4MB/s]
Loaded rows: 6037
Tokenizing...
tokenizing: 100%|██████████████████████████| 6037/6037 [00:06<00:00, 946.48it/s]
Tokenized examples : 6037
Skipped examples   : 0
Min length         : 88
Avg length         : 210.4
Max length         : 584
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 50.9kB [00:00

In [15]:
from pathlib import Path

path = Path("/kaggle/working/02_train_fast_visual.py")
text = path.read_text(encoding="utf-8")

# Add PeftModel import
text = text.replace(
    "from peft import (\n    LoraConfig,\n    get_peft_model,\n    prepare_model_for_kbit_training,\n)",
    "from peft import (\n    LoraConfig,\n    get_peft_model,\n    prepare_model_for_kbit_training,\n    PeftModel,\n)"
)

# Add adapter_path argument
text = text.replace(
    'parser.add_argument("--resume_from_checkpoint", type=str, default=None)',
    'parser.add_argument("--resume_from_checkpoint", type=str, default=None)\n    parser.add_argument("--adapter_path", type=str, default=None)'
)

# Replace LoRA creation block
old = '''    model = get_peft_model(model, lora_config)

    if is_main_process():
        model.print_trainable_parameters()
'''

new = '''    if args.adapter_path:
        if is_main_process():
            print(f"Loading existing adapter from: {args.adapter_path}")
        model = PeftModel.from_pretrained(
            model,
            args.adapter_path,
            is_trainable=True,
        )
    else:
        model = get_peft_model(model, lora_config)

    if is_main_process():
        model.print_trainable_parameters()
'''

text = text.replace(old, new)

path.write_text(text, encoding="utf-8")
print("Patched adapter continuation support.")

Patched adapter continuation support.


In [16]:
!python -m py_compile /kaggle/working/02_train_fast_visual.py

In [17]:
!rm -rf qlora_6000_cont
!CUDA_VISIBLE_DEVICES=0,1 accelerate launch \
  --multi_gpu \
  --num_processes=2 \
  --num_machines=1 \
  --mixed_precision=fp16 \
  --dynamo_backend=no \
  02_train_fast_visual.py \
  --data train_verified.jsonl \
  --adapter_path qlora_6000_fast \
  --output_dir qlora_6000_cont \
  --num_train_epochs 0.5 \
  --batch_size 2 \
  --grad_accum 4 \
  --max_seq_len 768 \
  --lr 5e-5 \
  --logging_steps 5 \
  --save_steps 100 \
  --save_total_limit 3 \
  --report_to tensorboard \
  --run_name qwen35_08b_gsm8k_6000_continue_0p5epoch

Fast Visual QLoRA Training
Base model      : Qwen/Qwen3.5-0.8B
Data            : train_verified.jsonl
Output          : qlora_6000_cont
World size      : 2
Local rank      : 0
Epochs          : 0.5
Batch/GPU       : 2
Grad accum      : 4
Effective batch : 16
Max seq len     : 768
TensorBoard dir : runs
Loaded rows: 6037
Tokenizing...
tokenizing: 100%|██████████████████████████| 6037/6037 [00:06<00:00, 942.67it/s]
Tokenized examples : 6037
Skipped examples   : 0
Min length         : 88
Avg length         : 210.4
Max length         : 584
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
[transformers] The fast path is not available because one of t

In [18]:
!rm -rf qlora_6000_cont2
!CUDA_VISIBLE_DEVICES=0,1 accelerate launch \
  --multi_gpu \
  --num_processes=2 \
  --num_machines=1 \
  --mixed_precision=fp16 \
  --dynamo_backend=no \
  02_train_fast_visual.py \
  --data train_verified.jsonl \
  --adapter_path qlora_6000_cont \
  --output_dir qlora_6000_cont2 \
  --num_train_epochs 0.5 \
  --batch_size 2 \
  --grad_accum 4 \
  --max_seq_len 768 \
  --lr 2e-5 \
  --logging_steps 5 \
  --save_steps 100 \
  --save_total_limit 3 \
  --report_to tensorboard \
  --run_name qwen35_08b_gsm8k_6000_continue_2

Fast Visual QLoRA Training
Base model      : Qwen/Qwen3.5-0.8B
Data            : train_verified.jsonl
Output          : qlora_6000_cont2
World size      : 2
Local rank      : 0
Epochs          : 0.5
Batch/GPU       : 2
Grad accum      : 4
Effective batch : 16
Max seq len     : 768
TensorBoard dir : runs
Loaded rows: 6037
Tokenizing...
tokenizing: 100%|██████████████████████████| 6037/6037 [00:06<00:00, 929.04it/s]
Tokenized examples : 6037
Skipped examples   : 0
Min length         : 88
Avg length         : 210.4
Max length         : 584
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
[transformers] The fast path is not available because one of 

FINAL MERGE

In [19]:
import torch, gc
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = "Qwen/Qwen3.5-0.8B"
ADAPTER_DIR = "qlora_6000_cont2"
MERGED_DIR = "merged_fp16_6000_cont2"

gc.collect()
torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="cpu",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(base, ADAPTER_DIR, device_map="cpu")
model = model.merge_and_unload()
model.save_pretrained(MERGED_DIR, safe_serialization=True, max_shard_size="2GB")

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)
tokenizer.save_pretrained(MERGED_DIR)

print("Merged model saved to:", MERGED_DIR)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to: merged_fp16_6000_cont2


In [20]:
from pathlib import Path

path = Path("/kaggle/working/04_evaluate_visual.py")
text = path.read_text(encoding="utf-8")

lines = [
    line for line in text.splitlines()
    if not line.strip().startswith("%%writefile")
]

path.write_text("\n".join(lines) + "\n", encoding="utf-8")

!python -m py_compile /kaggle/working/04_evaluate_visual.py

In [21]:
!python 04_evaluate_visual.py \
  --mode baseline \
  --model Qwen/Qwen3.5-0.8B \
  --n 300 \
  --batch_size 8 \
  --out eval_baseline_300.json \
  --plots_dir eval_plots_6000_cont

README.md: 7.93kB [00:00, 24.1MB/s]
main/train-00000-of-00001.parquet: 100%|███| 2.31M/2.31M [00:00<00:00, 5.45MB/s]
main/test-00000-of-00001.parquet: 100%|██████| 419k/419k [00:00<00:00, 1.98MB/s]
Generating test split: 100%|█████| 1319/1319 [00:00<00:00, 269302.78 examples/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
HF batched eval: 100%|██████████████████████████| 38/38 [13:57<00:00, 22.03s/it]

Mode      : baseline
Model     : Qwen/Qwen3.5-0.8B
N         : 300
Accuracy  : 32.33%
95% CI    : [27.33%, 38.00%]

Saved predictions to: eval_baseline_300_predictions.jsonl
Saved plots to: eval_plots_6000_cont


In [22]:
!python 04_evaluate_visual.py \
  --mode distilled \
  --model merged_fp16_6000_cont2 \
  --n 300 \
  --batch_size 8 \
  --out eval_distilled_6000_cont2_300.json \
  --plots_dir eval_plots_6000_cont2

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
HF batched eval: 100%|██████████████████████████| 38/38 [08:05<00:00, 12.78s/it]

Mode      : distilled
Model     : merged_fp16_6000_cont2
N         : 300
Accuracy  : 50.33%
95% CI    : [45.00%, 56.33%]

Saved predictions to: eval_distilled_6000_cont2_300_predictions.jsonl
Saved plots to: eval_plots_6000_cont2


In [23]:
!cat eval_baseline_300.json
!cat eval_distilled_6000_cont2_300.json

{
  "mode": "baseline",
  "model": "Qwen/Qwen3.5-0.8B",
  "n": 300,
  "accuracy": 0.3233,
  "ci_lo": 0.2733,
  "ci_hi": 0.38,
  "correct": 97,
  "wrong": 203
}{
  "mode": "distilled",
  "model": "merged_fp16_6000_cont2",
  "n": 300,
  "accuracy": 0.5033,
  "ci_lo": 0.45,
  "ci_hi": 0.5633,
  "correct": 151,
  "wrong": 149
}

In [24]:
!zip -r qwen_gsm8k_6000_cont2_final.zip \
  qlora_6000_cont2 \
  merged_fp16_6000_cont2 \
  eval_baseline_300.json \
  eval_distilled_6000_cont2_300.json \
  eval_distilled_6000_cont2_300_predictions.jsonl \
  eval_plots_6000_cont2

  adding: qlora_6000_cont2/ (stored 0%)
  adding: qlora_6000_cont2/tokenizer_config.json (deflated 65%)
  adding: qlora_6000_cont2/README.md (deflated 65%)
  adding: qlora_6000_cont2/adapter_model.safetensors (deflated 7%)
  adding: qlora_6000_cont2/tokenizer.json (deflated 80%)
  adding: qlora_6000_cont2/runs/ (stored 0%)
  adding: qlora_6000_cont2/runs/Jul07_17-52-50_50c3c4bc069e/ (stored 0%)
  adding: qlora_6000_cont2/runs/Jul07_17-52-50_50c3c4bc069e/events.out.tfevents.1783446770.50c3c4bc069e.461.0 (deflated 65%)
  adding: qlora_6000_cont2/adapter_config.json (deflated 59%)
  adding: qlora_6000_cont2/checkpoint-189/ (stored 0%)
  adding: qlora_6000_cont2/checkpoint-189/tokenizer_config.json (deflated 65%)
  adding: qlora_6000_cont2/checkpoint-189/README.md (deflated 65%)
  adding: qlora_6000_cont2/checkpoint-189/scaler.pt (deflated 64%)
  adding: qlora_6000_cont2/checkpoint-189/optimizer.pt (deflated 9%)
  adding: qlora_6000_cont2/checkpoint-189/adapter_model.safetensors (deflated 

In [25]:
from IPython.display import FileLink
FileLink("qwen_gsm8k_6000_cont2_final.zip")

/kaggle/working/qwen_gsm8k_6000_cont2_final.zip